# Loading Data

In [86]:
import numpy as np
import pandas as pd

In [87]:
# USE THIS WHEN RUNNING LOCALLY

df_1 = pd.read_csv(r"/data/bbb/cricsheet_ipl_bbb.csv")
df_2 = pd.read_csv(r"/data/bbb/ganjoo_ipl_bbb.csv")
player_dict = pd.read_csv(r"/data/register_data/player_details.csv")

Cleaning Data to merge together

In [88]:
bowl_dict = {
    'BAW Mendis': 'Right Arm Off Spin',
    'MM Patel': 'Right Arm Fast Medium',
    'AN Ahmed': 'Right Arm Fast Medium',
    'AUK Pathan': 'Right Arm Medium',
    'SB Wagh': 'Left arm Medium fast',
    'MM Sharma': 'Right Arm Fast Medium',
    'AR Patel': 'Left Arm Orthodox',
    'BE Hendricks': 'Left Arm Fast Medium'
}

# Fill null values in 'bowler_bowl_style' by mapping 'bowler' to 'bowler_bowl_style'
df_1['bowler_bowl_style'] = df_1.apply(
    lambda row: bowl_dict.get(row['bowler'], row['bowler_bowl_style']) if pd.isnull(row['bowler_bowl_style']) else row['bowler_bowl_style'],
    axis=1
)

bat_dict = {
    'AS Yadav': 'Right Hand Bat',
    'DT Patil': 'Right Hand Bat',
    'MM Patel': 'Right Hand Bat',
    'BAW Mendis': 'Right Hand Bat',
    'AN Ahmed': 'Right Hand Bat',
    'RS Sodhi': 'Right Hand Bat',
    'AUK Pathan': 'Right Hand Bat',
    'SB Wagh': 'Left Hand Bat',
    'MM Sharma': 'Right Hand Bat',
    'AR Patel': 'Left Hand Bat',
    'BE Hendricks': 'Left Hand Bat',
}

# Fill null values in 'batter_bat_style' by mapping 'batter' to 'batter_bat_style'
df_1['batter_bat_style'] = df_1.apply(
    lambda row: bat_dict.get(row['batter'], row['batter_bat_style']) if pd.isnull(row['batter_bat_style']) else row['batter_bat_style'],
    axis=1
)

id_map = dict(zip(player_dict['cricsheet_name'], player_dict['key_cricinfo']))
df_1['batter_id'] = df_1['batter'].map(id_map)
df_1['bowler_id'] = df_1['bowler'].map(id_map)

missing_id = {
    'Niraj Patel': 32240,
    'Harmeet Singh (2)': 391128
}

# Fill missing values in 'batter_id' and 'bowler_id' using the mapping
df_1['batter_id'] = df_1.apply(
    lambda row: missing_id.get(row['batter'], row['batter_id']) if pd.isnull(row['batter_id']) else row['batter_id'],
    axis=1
)
df_1['bowler_id'] = df_1.apply(
    lambda row: missing_id.get(row['bowler'], row['bowler_id']) if pd.isnull(row['bowler_id']) else row['bowler_id'],
    axis=1
)

csheet_to_cbuzz =  dict(zip(player_dict['unique_name'], player_dict['cricbuzz_name']))
csheet_to_cinfo = dict(zip(player_dict['unique_name'], player_dict['key_cricinfo']))
df_1['batter'] = df_1['batter'].map(csheet_to_cbuzz)
df_1['bowler'] = df_1['bowler'].map(csheet_to_cbuzz)
df_1['player_dismissed'] = df_1['player_dismissed'].map(csheet_to_cinfo)

In [89]:
cols = ['extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty']
for col in cols:
    df_1[col] = df_1[col].fillna(0).astype(int)

df_1['bowlruns'] = df_1['runs_off_bat'] + df_1['wides'] + df_1['noballs']
df_1['ballfaced'] = df_1['wides'].apply(lambda x: 0 if x != 0 else 1)

In [90]:
df_1.drop(columns=['penalty', 'other_player_dismissed', 'other_wicket_type', 'batter_role', 'bowler_role', 'extras'], inplace=True)

df_2.drop(columns=[
    'ball', 'outcome', 'score', 'out', 'over', 'cur_bat_runs',
    'cur_bat_bf', 'cur_bowl_ovr', 'cur_bowl_wkts', 'cur_bowl_runs',
    'inns_runs', 'inns_wkts', 'inns_balls', 'inns_runs_rem',
    'inns_balls_rem', 'inns_rr', 'inns_rrr', 'target', 'max_balls',
    'year', 'country', 'winner', 'toss', 'competition',
    'bat_out', 'wagonX', 'wagonY', 'wagonZone', 'line',
    'length', 'shot', 'control', 'predscore', 'wprob', 'bowl_kind'], inplace=True)

In [91]:
df_1.rename(columns={'start_date':'date', 'innings':'inns', 'batting_team':'team_bat', 'bowling_team':'team_bowl',
                     'runs_off_bat':'batruns', 'batter_bat_style':'bat_hand', 'bowler_bowl_style':'bowl_style'}, inplace=True)

df_2.rename(columns={'p_match':'match_id', 'p_bat':'batter_id', 'p_bowl':'bowler_id', 'bat':'batter', 'bowl':'bowler', 'ball_id':'ball',
                     'dismissal':'wicket_type', 'p_out':'player_dismissed','noball':'noballs', 'wide':'wides', 'ground':'venue'}, inplace=True)

df_1 = df_1.reindex(columns=[
    'date', 'match_id', 'inns', 'ball', 'batter', 'batter_id', 'team_bat',
    'bowler', 'bowler_id', 'team_bowl', 'batruns', 'bowlruns', 'ballfaced', 'wides', 'noballs', 'byes',
    'legbyes', 'wicket_type', 'player_dismissed', 'bat_hand', 'bowl_style', 'venue'])

df_2 = df_2.reindex(columns=[
    'date', 'match_id', 'inns', 'ball', 'batter', 'batter_id', 'team_bat',
    'bowler', 'bowler_id', 'team_bowl', 'batruns', 'bowlruns', 'ballfaced', 'wides', 'noballs', 'byes',
    'legbyes', 'wicket_type', 'player_dismissed', 'bat_hand', 'bowl_style', 'venue'])

In [92]:
print(df_1.columns)
print(df_2.columns)

Index(['date', 'match_id', 'inns', 'ball', 'batter', 'batter_id', 'team_bat',
       'bowler', 'bowler_id', 'team_bowl', 'batruns', 'bowlruns', 'ballfaced',
       'wides', 'noballs', 'byes', 'legbyes', 'wicket_type',
       'player_dismissed', 'bat_hand', 'bowl_style', 'venue'],
      dtype='object')
Index(['date', 'match_id', 'inns', 'ball', 'batter', 'batter_id', 'team_bat',
       'bowler', 'bowler_id', 'team_bowl', 'batruns', 'bowlruns', 'ballfaced',
       'wides', 'noballs', 'byes', 'legbyes', 'wicket_type',
       'player_dismissed', 'bat_hand', 'bowl_style', 'venue'],
      dtype='object')


In [93]:
# Fixing data types

df_1['bowler_id'] = df_1['bowler_id'].astype(np.int64)
df_1['wides'] = df_1['wides'].astype(np.int64)
df_1['noballs'] = df_1['noballs'].astype(np.int64)
df_1['byes'] = df_1['byes'].astype(np.int64)
df_1['legbyes'] = df_1['legbyes'].astype(np.int64)
df_2['player_dismissed'] = df_2.apply(
    lambda row: np.nan if pd.isnull(row['wicket_type']) else row['player_dismissed'],
    axis=1
)

Concatenating

In [94]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108850 entries, 0 to 108849
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   date              108850 non-null  object 
 1   match_id          108850 non-null  int64  
 2   inns              108850 non-null  int64  
 3   ball              108850 non-null  float64
 4   batter            108850 non-null  object 
 5   batter_id         108850 non-null  int64  
 6   team_bat          108850 non-null  object 
 7   bowler            108850 non-null  object 
 8   bowler_id         108850 non-null  int64  
 9   team_bowl         108850 non-null  object 
 10  batruns           108850 non-null  int64  
 11  bowlruns          108850 non-null  int64  
 12  ballfaced         108850 non-null  int64  
 13  wides             108850 non-null  int64  
 14  noballs           108850 non-null  int64  
 15  byes              108850 non-null  int64  
 16  legbyes           10

In [95]:
df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151983 entries, 0 to 151982
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   date              151983 non-null  object 
 1   match_id          151983 non-null  int64  
 2   inns              151983 non-null  int64  
 3   ball              151983 non-null  float64
 4   batter            151983 non-null  object 
 5   batter_id         151983 non-null  int64  
 6   team_bat          151983 non-null  object 
 7   bowler            151983 non-null  object 
 8   bowler_id         151983 non-null  int64  
 9   team_bowl         151983 non-null  object 
 10  batruns           151983 non-null  int64  
 11  bowlruns          151983 non-null  int64  
 12  ballfaced         151983 non-null  int64  
 13  wides             151983 non-null  int64  
 14  noballs           151983 non-null  int64  
 15  byes              151983 non-null  int64  
 16  legbyes           15

In [96]:
combined_df = pd.concat([df_1, df_2], ignore_index=True)

Fixing batting and bowling styles

In [97]:
bat_hand_dict = {
    ' Left Handed Bat ': 'LHB',
    ' Right Handed Bat ': 'RHB',
    'Right Hand Bat': 'RHB',
    'Left Hand Bat' : 'LHB',
    'RHB': 'RHB',
    'LHB': 'LHB',
    ' -- ' : 'LHB' # This is a hard coded fix as there is only one batsman whose hand is --
}

combined_df['bat_hand'] = combined_df['bat_hand'].map(bat_hand_dict)

bowl_style_dict = {
    ' Right-arm fast-medium ':'RAF',
    ' Left-arm fast-medium ': 'LAF',
    ' Left-arm orthodox ' : 'SLA',
    ' Right-arm legbreak ' : 'RLB',
    ' Right-arm medium ' : 'RAF',
    ' Right-arm fast ' : 'RAF',
    ' Right-arm offbreak ' : 'ROB',
    'Right Arm Fast Medium' : 'RAF',
    'Right Arm Off Spin' : 'ROB',
    ' Left-arm fast ' : 'LAF',
    'Left arm Medium fast' : 'LAF',
    'Right Arm Medium' : 'RAF',
    ' Left-arm medium ' : 'LAF',
    ' Left-arm chinaman ' : 'LAF',
    'Left Arm Orthodox' : 'SLA',
    'Left Arm Fast Medium' : 'LAF',
    'RF' : 'RAF',
    'OB' : 'ROB',
    'LFM' : 'LAF',
    'LF' : 'LAF',
    'RM' : 'RAF',
    'RMF' : 'RAF',
    'RFM' : 'RAF',
    'LBG' : 'RLB',
    'LWS' : 'LWS',
    'LMF' : 'LAF',
    'SLA' : 'SLA',
    'LB' : 'RLB',
    'LM' : 'LAF',
    'OB/LB' : 'ROB',
    'OB/LBG' : 'ROB',
    'RM/OB' : 'ROB',
    'SLA/LWS' : 'LWS',
    'RM/OB/LB' : 'RAF'
}

combined_df['bowl_style'] = combined_df['bowl_style'].map(bowl_style_dict)

In [98]:
null_bowl_style_map = {
    'Karan Goel': 'RAF',
    'Wilkin Mota': 'RAF',
    'Palani Amarnath': 'RAF',
    'Dinesh Salunkhe': 'RLB',
    'Yo Mahesh': 'RAF',
    'Vikrant Yeligati': 'ROB',
    'Musavir Khote': 'RAF',
    'Doddapaneni Kalyankrishna': 'RAF',
    'V Singh': 'RAF',
    'Gagandeep Singh': 'RAF',
    'Paidikalva Vijaykumar': 'RAF',
    'Dominic Thornely': 'RAF',
    'Sarvesh Kumar': 'RAF',
    'Dwaraka Ravi Teja': 'RAF',
    'Kamran Khan': 'LAF',
    'Saleem Malik': 'ROB',
    'Ranadeb Bose': 'RAF',
    'Shoaib Ahmed': 'RAF',
    'Lee Carseldine': 'SLA',
    'Chetanya Nanda': 'RLB',
    'Yogesh Nagar': 'ROB',
    'Jaskaran Singh': 'RAF',
    'Rahil Shaikh': 'LAF',
    'Thisara Perera': 'RAF',
    'Mohnish Parmar': 'ROB',
    'Raiphi Gomez': 'RAF',
    'Arshad Bhatt': 'SLA',
    'Ryan Ninan': 'ROB',
    'Bharath Chipli': 'RAF',
    'Gnaneswara Rao': 'ROB',
    'Ben Rohrer': 'ROB',
    'Karnvir Singh': 'RLB',
    'Tirumalasetti Suman' : 'ROB'
}

# Fill null values in 'bowl_style' using the mapping
combined_df['bowl_style'] = combined_df.apply(
    lambda row: null_bowl_style_map.get(row['bowler'], row['bowl_style']) if pd.isnull(row['bowl_style']) else row['bowl_style'],
    axis=1
)

In [99]:
venue_dict = {
    'Feroz Shah Kotla': 'Arun Jaitley Stadium',
    'Sardar Patel Stadium': 'Narendra Modi Stadium',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 'Andhra Cricket Association-Visakhapatnam District Cricket Association Stadium',
    'Punjab Cricket Association Stadium':'Punjab Cricket Association IS Bindra Stadium',
    'Zayed Cricket Stadium': 'Sheikh Zayed Stadium',
    'Subrata Roy Sahara Stadium' : 'Maharashtra Cricket Association Stadium'
}
combined_df['venue'] = combined_df['venue'].str.split(',').str[0]
combined_df['venue'] = combined_df['venue'].replace(venue_dict)

combined_df.wicket_type.replace('leg before wicket', 'lbw', inplace=True)

In [100]:
combined_df.sample(10)

,date,match_id,inns,ball,batter,batter_id,team_bat,bowler,bowler_id,team_bowl,...,ballfaced,wides,noballs,byes,legbyes,wicket_type,player_dismissed,bat_hand,bowl_style,venue
182383,2017-04-10,1082598,1,14.03,Stuart Binny,27223,Royal Challengers Bangalore,Axar Patel,554691,Punjab Kings,...,1,0,0,0,0,NaN,NaN,RHB,SLA,Holkar Cricket Stadium
33308,2010-03-29,419131,2,2.30,Sourav Ganguly,28779,Kolkata Knight Riders,Dirk Nannes,6981,Delhi Daredevils,...,1,0,0,0,0,NaN,NaN,LHB,LAF,Arun Jaitley Stadium
133272,2021-10-13,1254116,1,5.01,Shikhar Dhawan,28235,Delhi Capitals,Lockie Ferguson,493773,Kolkata Knight Riders,...,1,0,0,0,0,NaN,NaN,LHB,RAF,Sharjah Cricket Stadium
31397,2010-03-23,419123,2,10.30,Murali Vijay,237095,Chennai Super Kings,Jacques Kallis,45789,Royal Challengers Bangalore,...,1,0,0,0,0,NaN,NaN,RHB,RAF,M Chinnaswamy Stadium
254315,2024-04-27,1426281,1,11.01,Shai Hope,581379,Delhi Capitals,Mohammad Nabi,25913,Mumbai Indians,...,1,0,0,0,0,NaN,NaN,RHB,ROB,Arun Jaitley Stadium
55878,2011-05-16,501259,2,12.60,Kumar Sangakkara,50710,Deccan Chargers,Mithun Manhas,31001,Pune Warriors,...,1,0,0,0,0,NaN,NaN,LHB,ROB,Dr DY Patil Sports Academy
125626,2022-04-17,1304074,1,18.05,Odean Smith,820691,Punjab Kings,Bhuvneshwar Kumar,326016,Sunrisers Hyderabad,...,1,0,0,0,0,NaN,NaN,RHB,RAF,Dr DY Patil Sports Academy
222415,2018-04-08,1136563,2,18.04,Vinay Kumar,35731,Kolkata Knight Riders,Kulwant Khejroliya,1083033,Royal Challengers Bangalore,...,1,0,0,0,0,NaN,NaN,RHB,LAF,Eden Gardens
161682,2015-04-20,829739,1,14.04,Yuvraj Singh,36084,Delhi Capitals,Piyush Chawla,32966,Kolkata Knight Riders,...,1,0,0,0,0,NaN,NaN,LHB,RLB,Arun Jaitley Stadium
14205,2009-04-19,392184,2,4.10,Herschelle Gibbs,45224,Deccan Chargers,Ashok Dinda,227712,Kolkata Knight Riders,...,1,0,0,0,0,NaN,NaN,RHB,RAF,Newlands


In [101]:
combined_df.to_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\bbb\combined_ipl_bbb.csv", index=False)